# Homework 13: Productization
Train, persist, serve, and call a two-feature regression model.

In [1]:
from pathlib import Path
import subprocess,sys,time
import joblib
import requests
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
P=Path.cwd().resolve(); P=P if P.name=='homework13' else P/'homework'/'homework13'

In [2]:
X,y=make_regression(n_samples=100,n_features=2,noise=.1,random_state=42)
model=LinearRegression().fit(X,y)
(P/'model').mkdir(exist_ok=True); joblib.dump(model,P/'model/model.pkl')
loaded=joblib.load(P/'model/model.pkl'); print('Reloaded-model prediction:',float(loaded.predict([[.25,-.4]])[0]))

Reloaded-model prediction: -7.698750717752957


`app.py` loads `model/model.pkl` once at module import, validates both POST JSON and GET path inputs, and returns JSON errors with HTTP 400.

In [3]:
server=subprocess.Popen([sys.executable,'app.py'],cwd=P,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
for _ in range(40):
 try:
  requests.get('http://127.0.0.1:5050/predict/0/0',timeout=.25); break
 except requests.RequestException: time.sleep(.25)
else: server.terminate(); raise RuntimeError('API did not start')
print('API server started')

API server started


In [4]:
post=requests.post('http://127.0.0.1:5050/predict',json={'features':[.25,-.4]},timeout=3)
get=requests.get('http://127.0.0.1:5050/predict/.25/-.4',timeout=3)
bad=requests.post('http://127.0.0.1:5050/predict',json={'features':[.25]},timeout=3)
print('POST',post.status_code,post.json())
print('GET ',get.status_code,get.json())
print('BAD ',bad.status_code,bad.json())
assert post.status_code==get.status_code==200 and bad.status_code==400
assert abs(post.json()['prediction']-get.json()['prediction'])<1e-12

POST 200 {'prediction': -7.698750717752957}
GET  200 {'prediction': -7.698750717752957}
BAD  400 {'error': 'features must be a list containing exactly two numbers'}


In [5]:
server.terminate(); server.wait(timeout=5); print('API server stopped after tests')

API server stopped after tests


Both access patterns return the same model prediction, while malformed input returns a controlled 400 response. The visible outputs above are end-to-end evidence that serialization, startup loading, routing, validation, and client calls work together.